# BM25: The Foundation of Modern Search

BM25 (Best Matching 25) is a ranking function used by search engines to estimate the relevance of documents to a given search query. Despite being developed in the 1990s, it remains a core component of modern search systems including Elasticsearch, and is often combined with neural approaches in hybrid retrieval.

In this notebook, we'll implement BM25 from scratch to understand exactly how it works.

## The BM25 Formula

BM25 scores a document `D` for a query `Q` containing terms `q1, q2, ..., qn` as:

$$\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{\text{avgdl}})}$$

Where:
- `f(qi, D)` = frequency of term `qi` in document `D`
- `|D|` = length of document `D` (in words)
- `avgdl` = average document length in the corpus
- `k1` = term frequency saturation parameter (typically 1.2-2.0)
- `b` = length normalization parameter (typically 0.75)

The IDF (Inverse Document Frequency) component is:

$$\text{IDF}(q_i) = \ln\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1\right)$$

Where:
- `N` = total number of documents
- `n(qi)` = number of documents containing term `qi`

In [ ]:
import math
import re
from collections import Counter
from dataclasses import dataclass

## Step 1: Tokenization

Before we can compute BM25, we need to tokenize our documents. We'll use a simple whitespace + punctuation tokenizer.

In [ ]:
def tokenize(text: str) -> list[str]:
    """Simple tokenizer: lowercase, split on non-alphanumeric, filter short tokens."""
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    return [t for t in tokens if len(t) > 1]  # Filter single characters

# Test it
sample = "The quick brown fox jumps over the lazy dog."
print(tokenize(sample))

## Step 2: BM25 Implementation

Now let's implement BM25 as a class that can index documents and score queries.

In [ ]:
@dataclass
class BM25:
    """BM25 ranking implementation from scratch."""
    k1: float = 1.5  # Term frequency saturation
    b: float = 0.75  # Length normalization
    
    def __post_init__(self):
        self.corpus = []
        self.doc_freqs = []  # Term frequencies per document
        self.doc_lengths = []
        self.avgdl = 0
        self.idf = {}  # IDF scores per term
        self.n_docs = 0
    
    def index(self, documents: list[str]):
        """Index a corpus of documents."""
        self.corpus = documents
        self.n_docs = len(documents)
        
        # Tokenize and compute term frequencies
        self.doc_freqs = []
        self.doc_lengths = []
        doc_containing_term = Counter()  # How many docs contain each term
        
        for doc in documents:
            tokens = tokenize(doc)
            self.doc_lengths.append(len(tokens))
            
            freq = Counter(tokens)
            self.doc_freqs.append(freq)
            
            # Count unique terms in this doc
            for term in set(tokens):
                doc_containing_term[term] += 1
        
        # Compute average document length
        self.avgdl = sum(self.doc_lengths) / self.n_docs if self.n_docs > 0 else 0
        
        # Compute IDF for each term
        self.idf = {}
        for term, n in doc_containing_term.items():
            # BM25 IDF formula
            self.idf[term] = math.log((self.n_docs - n + 0.5) / (n + 0.5) + 1)
        
        print(f"Indexed {self.n_docs} documents")
        print(f"Average document length: {self.avgdl:.1f} tokens")
        print(f"Vocabulary size: {len(self.idf)} terms")
    
    def score(self, query: str, doc_idx: int) -> float:
        """Score a single document against a query."""
        query_terms = tokenize(query)
        doc_freq = self.doc_freqs[doc_idx]
        doc_len = self.doc_lengths[doc_idx]
        
        score = 0.0
        for term in query_terms:
            if term not in self.idf:
                continue  # Term not in corpus
            
            tf = doc_freq.get(term, 0)  # Term frequency in this doc
            idf = self.idf[term]
            
            # BM25 scoring formula
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            score += idf * (numerator / denominator)
        
        return score
    
    def search(self, query: str, top_k: int = 5) -> list[tuple[int, float, str]]:
        """Search the corpus and return top-k results."""
        scores = [(i, self.score(query, i)) for i in range(self.n_docs)]
        scores.sort(key=lambda x: x[1], reverse=True)
        
        results = []
        for doc_idx, score in scores[:top_k]:
            if score > 0:
                results.append((doc_idx, score, self.corpus[doc_idx][:100] + "..."))
        return results

## Step 3: Testing BM25

Let's test our implementation with a small corpus.

In [ ]:
# Sample corpus about programming topics
documents = [
    "Python is a high-level programming language known for its simplicity and readability. It supports multiple programming paradigms.",
    "Machine learning is a subset of artificial intelligence that enables systems to learn from data without being explicitly programmed.",
    "Neural networks are computing systems inspired by biological neural networks. They are used in deep learning applications.",
    "Natural language processing (NLP) is a field of AI that focuses on the interaction between computers and human language.",
    "Database systems store and organize data. SQL is a common language for querying relational databases.",
    "Web development involves building websites and web applications using HTML, CSS, JavaScript, and backend technologies.",
    "APIs allow different software systems to communicate with each other. REST and GraphQL are popular API architectures.",
    "Version control systems like Git help developers track changes in code and collaborate effectively on projects.",
    "Cloud computing provides on-demand computing resources over the internet. AWS, Azure, and GCP are major providers.",
    "Cybersecurity involves protecting computer systems and networks from digital attacks and unauthorized access.",
]

# Index the corpus
bm25 = BM25(k1=1.5, b=0.75)
bm25.index(documents)

In [ ]:
# Search for different queries
queries = [
    "machine learning artificial intelligence",
    "programming language",
    "database SQL",
    "neural networks deep learning",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")
    
    results = bm25.search(query, top_k=3)
    for rank, (doc_idx, score, snippet) in enumerate(results, 1):
        print(f"\n{rank}. [Score: {score:.3f}]")
        print(f"   {snippet}")

## Understanding IDF Scores

IDF gives higher weight to rare terms. Let's examine which terms have the highest and lowest IDF scores in our corpus.

In [ ]:
# Sort terms by IDF
sorted_idf = sorted(bm25.idf.items(), key=lambda x: x[1], reverse=True)

print("Highest IDF (rare terms - most discriminative):")
for term, idf in sorted_idf[:10]:
    print(f"  {term}: {idf:.3f}")

print("\nLowest IDF (common terms - least discriminative):")
for term, idf in sorted_idf[-10:]:
    print(f"  {term}: {idf:.3f}")

## The Effect of k1 and b Parameters

- **k1** controls term frequency saturation. Higher values mean term frequency has more impact on the score.
- **b** controls length normalization. b=1 means full length normalization, b=0 means no normalization.

Let's see how these parameters affect scoring.

In [ ]:
query = "programming language"

# Test different parameter combinations
configs = [
    (1.2, 0.75, "Default (k1=1.2, b=0.75)"),
    (2.0, 0.75, "Higher k1 (k1=2.0, b=0.75)"),
    (1.2, 0.0, "No length norm (k1=1.2, b=0.0)"),
    (1.2, 1.0, "Full length norm (k1=1.2, b=1.0)"),
]

print(f"Query: '{query}'\n")

for k1, b, desc in configs:
    bm25_test = BM25(k1=k1, b=b)
    bm25_test.index(documents)
    
    results = bm25_test.search(query, top_k=3)
    print(f"\n{desc}:")
    for rank, (doc_idx, score, _) in enumerate(results, 1):
        print(f"  {rank}. Doc {doc_idx}: {score:.3f}")

## Limitations of BM25

BM25 is powerful but has limitations:

1. **No semantic understanding**: "car" and "automobile" are treated as completely different terms
2. **Exact matching only**: Misspellings or variations won't match
3. **Bag of words**: Word order and context are ignored

Let's demonstrate the semantic gap:

In [ ]:
# These should find similar documents but won't with BM25
semantic_queries = [
    ("machine learning", "AI that learns from data"),
    ("programming language", "coding syntax"),
    ("web development", "building websites"),
]

print("Semantic gap demonstration:")
print("="*60)

for exact, semantic in semantic_queries:
    print(f"\nExact: '{exact}'")
    exact_results = bm25.search(exact, top_k=1)
    if exact_results:
        print(f"  -> Doc {exact_results[0][0]}, Score: {exact_results[0][1]:.3f}")
    
    print(f"Semantic: '{semantic}'")
    semantic_results = bm25.search(semantic, top_k=1)
    if semantic_results:
        print(f"  -> Doc {semantic_results[0][0]}, Score: {semantic_results[0][1]:.3f}")
    else:
        print(f"  -> No results (terms not in corpus!)")

---

## Exercises

### Exercise 1: Implement BM25+

BM25+ is a variant that addresses the issue where BM25 can give negative IDF scores for very common terms. Modify the IDF formula to use:

$$\text{IDF}_{+}(q_i) = \max\left(0, \ln\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5}\right)\right)$$

In [ ]:
# Your implementation here
class BM25Plus(BM25):
    """BM25+ with non-negative IDF scores."""
    pass  # Implement the modified IDF formula

### Exercise 2: Add Stopword Removal

Implement stopword removal in the tokenizer. Common stopwords like "the", "is", "and" don't help with retrieval.

In [ ]:
STOPWORDS = {"the", "is", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by"}

def tokenize_with_stopwords(text: str) -> list[str]:
    """Tokenize with stopword removal."""
    # Your implementation here
    pass

### Exercise 3: Query Term Weighting

Modify the search function to accept optional term weights. For example, the user might want to emphasize certain terms:

```python
bm25.search("machine learning neural", weights={"neural": 2.0})
```

In [ ]:
# Your implementation here
def weighted_search(bm25: BM25, query: str, weights: dict = None, top_k: int = 5):
    """Search with optional term weights."""
    pass

---

## Summary

- BM25 is a probabilistic ranking function based on term frequency and inverse document frequency
- The `k1` parameter controls term frequency saturation
- The `b` parameter controls document length normalization
- BM25 excels at exact keyword matching but lacks semantic understanding
- Modern search systems often combine BM25 with neural embeddings (hybrid retrieval) to get the best of both worlds

In the next notebook, we'll explore semantic search using embeddings, and then combine them in a hybrid approach.